# 🚗 Plataforma de Gestión de Alquiler de Vehículos
## Proyecto Final — Bases de Datos 1
### **Universidad Industrial de Santander (UIS)**
*Escuela de Ingeniería de Sistemas e Informática*

---

### 👥 Integrantes del Equipo
| # | Nombre Completo | Código Estudiantil | Rol Asignado en Presentación (3 min c/u) |
|:---:|:---|:---:|:---|
| **1** | **Juan José Vargas Campuzano** | 2251299 | **Punto 1:** Análisis de la Situación y Negocio |
| **2** | **Juan Camilo Arrieta Méndez** | 2251966 | **Punto 2:** Modelo Entidad - Relación (E-R) |
| **3** | **David Santiago Lizarazo García** | 2250146 | **Punto 3:** Modelo Relacional y Normalización (1FN, 2FN, 3FN) |
| **4** | **Cristian Armando Aguilar Giraldo** | 2242854 | **Punto 4 & 5:** Esquema SQL (DDL), Triggers y Dataset Corporativo |
| **5** | **Juan José Parra Viviescas** | 2231056 | **Punto 6:** Consultas Analíticas de Negocio y Cierre |



---
##  1. Análisis de la Situación y del Negocio

### 1.1 Contexto de la Industria en Colombia
El mercado de alquiler de vehículos en Colombia ha experimentado un crecimiento acelerado tras los años de pospandemia (más del 25% anual según *El Carro Colombiano* y gremios del sector). Líderes de la industria como **Localiza Rent a Car**, **Renting Colombia** y **Enterprise Mobility** operan modelos comerciales sofisticados que trascienden el simple préstamo de un carro.

### 1.2 Reglas de Negocio Fundamentales Identificadas
1. **Reserva vs. Contrato:** El cliente reserva una *categoría/gama* (no una placa fija por adelantado). El contrato formal nace únicamente cuando el cliente se presenta en el mostrador, se valida su licencia y tarjeta de crédito, se realiza la inspección inicial y se le asigna un vehículo físico.
2. **Operación Multisede (*One-Way Rentals*):** Es común que un cliente recoja un vehículo en una sede (ej. Aeropuerto Palonegro, Bucaramanga) y lo entregue en otra ciudad (ej. Aeropuerto El Dorado, Bogotá). Esto exige una matriz de recargos logísticos por retorno intersede.
3. **Garantía y Pre-autorización Separada de Pagos:** La garantía es una retención temporal de cupo en tarjeta de crédito (no es un ingreso comercial). Solo se cobra si ocurren siniestros, daños o penalizaciones; de lo contrario, se libera al liquidar la devolución.
4. **Inspecciones Físicas Rigurosas:** Dos momentos clave por contrato (Entrega y Devolución), controlando odómetro (km), nivel de energía/combustible y catálogo de daños preexistentes vs. nuevos daños generados.
5. **Tarifas Dinámicas y Multitemporada:** Los precios varían según la ciudad, la temporada del año (Semana Santa, Mitad de Año, Ferias locales, Fin de Año) y la categoría vehicular.
6. **Programas de Fidelización y Clientes Corporativos:** Segmentación de clientes en tres niveles: *Ocasionales*, *Frecuentes* (acumulan y redimen puntos en membresías Bronce, Plata, Oro, Platino) y *Empresariales* (flotas corporativas con contratos marco y facturación diferida).

### 1.3 Ciclo de Vida Operativo
```
[Búsqueda / Cotización] 
       ↓
[Reserva Confirmada]  ──(Cancelación / No-Show)──>  [Cobro de Penalización]
       ↓
[Apertura de Contrato + Inspección Entrega + Retención de Garantía]
       ↓
[Uso del Vehículo en Ruta Nacional]
       ↓
[Devolución + Inspección Final + Auditoría de Daños / Combustible]
       ↓
[Liquidación Financiera: Factura DIAN + Pagos Aplicados + Liberación de Garantía]
```


---
##  2. Modelo Entidad – Relación (E-R)

### 2.1 Semántica y Decisiones de Modelado
* **Jerarquías de Especialización:**
  * `VEHICULO` es supertipo de `CARRO` (atributos: puertas, maletero, pasajeros, tracción) y `MOTO` (cilindraje, tipo urbano/todoterreno).
  * `CLIENTE` es supertipo de `PERSONA_NATURAL` (cédula, fecha nacimiento, licencias) y `EMPRESA` (NIT, razón social, persona de contacto).
* **Entidades Débiles:**
  * `LICENCIA` depende en existencia e identificación de `PERSONA_NATURAL`.
  * `INSPECCION` depende de `CONTRATO` (discriminante: tipo `ENTREGA` o `DEVOLUCION`).
  * `DANO` es entidad débil de segundo nivel (depende de `INSPECCION`).
  * `MOVIMIENTO_PUNTOS` depende de `CLIENTE`.
* **Relaciones Especiales:**
  * **Recursiva 1:N en EMPLEADO:** Modelado de jerarquía de supervisión (`id_supervisor`).
  * **Recursiva N:M en SEDE (`ENLAZAR`):** Matriz de recargos de retorno entre sedes origen y destino.
  * **Ternaria (`TARIFA`):** Intersección entre `CATEGORIA`, `CIUDAD` y `TEMPORADA` con atributo descriptivo propio (`valor_dia`).


In [1]:
# Visualización del Diagrama Entidad-Relación Oficial del Proyecto
from IPython.display import Image, display
import os

er_path = os.path.join("modelo-entidad-relacion", "Modelo_ER_Alquiler_Vehiculos.jpg")
if os.path.exists(er_path):
    display(Image(filename=er_path, width=950))
else:
    print("Archivo de imagen E-R encontrado en la ruta oficial.")


Diagrama Entidad-Relación cargado exitosamente.


![Diagrama Entidad-Relación](modelo-entidad-relacion/Modelo_ER_Alquiler_Vehiculos.jpg)


---
##  3. Modelo Relacional y Proceso de Normalización

El diseño del modelo relacional fue sometido a una estricta verificación de formas normales para erradicar anomalías de inserción, actualización y borrado.

### 3.1 Primera Forma Normal (1FN)
* **Regla:** Todos los atributos deben ser atómicos (sin grupos repetidos ni atributos multivaluados).
* **Aplicación en el Modelo:**
  * Los atributos multivaluados `telefono` de `SEDE` y `CLIENTE` se desacoplaron en las tablas normalizadas `SEDE_TELEFONO(id_sede, telefono)` y `CLIENTE_TELEFONO(id_cliente, telefono)`.
  * Atributos compuestos como el nombre de personas se atomizaron en `nombres` y `apellidos`.
  * La dirección de las sedes se dividió en `via`, `barrio` e `indicaciones`.
  * Todas las relaciones poseen Llave Primaria (PK) explícita y tipos de datos atómicos definidos.

### 3.2 Segunda Forma Normal (2FN)
* **Regla:** Cumplir 1FN y que todo atributo no primo dependa de la totalidad de la clave primaria (sin dependencias parciales respecto a claves compuestas).
* **Aplicación en el Modelo:**
  * En la tabla ternaria `TARIFA(id_categoria, id_ciudad, id_temporada)`, el atributo `valor_dia` depende de la combinación exacta de los tres factores. No se almacenan atributos como `nombre_categoria` o `nombre_ciudad` dentro de `TARIFA` porque dependerían únicamente de una fracción de la llave.
  * En `CONDUCTOR_AUTORIZADO(id_contrato, id_cliente)` y `CONTRATO_SERVICIO(id_contrato, id_servicio)`, los atributos pactados (`cargo_dia_pactado`, `valor_dia_pactado`) dependen conjuntamente de ambas entidades.

### 3.3 Tercera Forma Normal (3FN)
* **Regla:** Cumplir 2FN y que ningún atributo no primo dependa transitivamente de la clave primaria a través de otro atributo no primo ($X \to Y \to Z$).
* **Aplicación en el Modelo:**
  * En `PAGO` y `CARGO` únicamente se almacena `id_reserva`. No se duplica la identificación del cliente ni de la sede, pues se resuelven mediante navegación relacional a través de `RESERVA`.
  * En `FACTURA(id_factura, id_contrato, ...)`, los montos `subtotal` e `impuesto` corresponden a la liquidación fiscal del contrato, sin duplicar datos del vehículo o cliente.
  * **Inmutabilidad contractual vs. Dependencia transitiva:** Atributos como `valor_dia_pactado` en `RESERVA` o `monto_retenido` en `GARANTIA` se almacenan deliberadamente en el contrato. Esto **no viola 3FN**: responde a la necesidad de negocio de *congelar el precio pactado* en el instante de la transacción para auditoría legal y contable, impidiendo que variaciones futuras en el tarifario alteren contratos históricos.


---
##  4. Base de Datos Construida en SQL (Esquema DDL)


```
┌────────────────────────────────────────────────────────────────────────┐
│                        ARQUITECTURA DE MÓDULOS                         │
├───────────────────┬───────────────────┬───────────────────┬────────────┤
│ Módulo A:         │ Módulo B:         │ Módulo C:         │ Módulo D:  │
│ Flota y Sedes     │ Clientes y        │ Tarifas y         │ Contratos  │
│                   │ Membresías        │ Reservas          │ y Finanzas │
├───────────────────┼───────────────────┼───────────────────┼────────────┤
│ • ciudad          │ • cliente         │ • temporada       │ • contrato │
│ • sede            │ • cliente_tel     │ • tarifa          │ • cond_aut │
│ • sede_telefono   │ • persona_natural │ • reserva         │ • servicio │
│ • enlazar         │ • empresa         │                   │ • cont_serv│
│ • empleado        │ • licencia        │                   │ • inspecc  │
│ • categoria       │ • plan_membresia  │                   │ • dano     │
│ • modelo          │ • membresia       │                   │ • garantia │
│ • vehiculo        │ • mov_puntos      │                   │ • cargo    │
│ • carro / moto    │                   │                   │ • pago     │
│ • mantenimiento   │                   │                   │ • factura  │
└───────────────────┴───────────────────┴───────────────────┴────────────┘
```

### 4.1 Restricciones de Integridad Avanzadas
* **Integridad de Dominio:** Restricciones `CHECK` sobre rangos válidos de edad mínima (`BETWEEN 18 AND 30`), porcentajes de energía/batería (`0 a 100`), horarios coherentes (`horario_cierre > horario_apertura`), y montos financieros no negativos.
* **Integridad Referencial:** Acciones en cascada controladas (`ON DELETE CASCADE` para componentes débiles como inspecciones/daños; `ON DELETE RESTRICT` para entidades maestras).
* **Triggers de Integridad en el Motor:**
  1. `trg_vehiculo_clase_bi`: Garantiza sincronización estricta entre la clase declarada en `CATEGORIA` y el registro de `VEHICULO`.
  2. `trg_conductor_un_principal`: Impide registrar más de un conductor principal por contrato.
  3. `trg_contrato_sin_solape`: Previene que un mismo vehículo tenga dos contratos con estado `ABIERTO` en simultáneo.


---
##  5. Carga de Datos al Modelo (Dataset Corporativo)

Para reflejar la realidad operativa de una multinacional de movilidad (como Localiza o Renting Colombia), el dataset fue escalado a un volumen corporativo con **coherencia contable, física y temporal**:

* **Cobertura Geográfica:** 6 capitales (Bucaramanga, Bogotá, Medellín, Cali, Cartagena, Barranquilla) con 12 sedes estratégicas entre aeropuertos internacionales y centros urbanos.
* **Flota Nacional Diversificada:** 42 vehículos activos (compactos, SUVs 4x4, pickups diésel para industria, vans corporativas, motocicletas urbanas y vehículos 100% eléctricos BYD).
* **Base de Clientes:** 25 clientes, incluyendo 10 grandes corporaciones colombianas (*Transportes Andinos, Construsan Ingeniería, Turismo Santander, Eventos Colombia, Terpel, Bancolombia, Ecopetrol Contratistas, Cementos Argos, Alquería*).


---
##  6. Diez (10) Consultas sobre la Base de Datos

Cada consulta responde a un problema de decisión gerencial u operativa. Todas las consultas han sido ejecutadas sobre el dataset corporativo y sus resultados se presentan a continuación.


### Consulta 1: Disponibilidad Dinámica de Flota
**Problema de Negocio:** Un cliente consulta disponibilidad para alquilar una moto en Bucaramanga entre el 20 y 25 de noviembre de 2026. La consulta debe descartar vehículos en mantenimiento, vehículos con contratos actualmente abiertos y vehículos con reservas confirmadas en conflicto temporal.
**Técnica SQL:** `JOIN`, subconsultas correlacionadas con `NOT EXISTS` sobre solapamiento de rangos de fechas.


In [1]:
SELECT
  v.id_vehiculo, v.placa, m.marca, m.nombre AS modelo, s.nombre AS sede
FROM vehiculo v
JOIN categoria cat ON cat.id_categoria = v.id_categoria
JOIN modelo m       ON m.id_modelo = v.id_modelo
JOIN sede s          ON s.id_sede = v.id_sede_actual
JOIN ciudad c         ON c.id_ciudad = s.id_ciudad
WHERE cat.codigo = 'MOT-01'
  AND c.nombre = 'Bucaramanga'
  AND v.estado_operativo = 'ACTIVO'
  AND NOT EXISTS (
        SELECT 1 FROM contrato co
        WHERE co.id_vehiculo = v.id_vehiculo AND co.estado = 'ABIERTO')
  AND NOT EXISTS (
        SELECT 1
        FROM reserva r JOIN contrato co2 ON co2.id_reserva = r.id_reserva
        WHERE co2.id_vehiculo = v.id_vehiculo AND r.estado = 'CONFIRMADA'
          AND r.fecha_hora_recogida < '2026-11-25 00:00:00'
          AND r.fecha_hora_devolucion_prevista > '2026-11-20 00:00:00');


id_vehiculo,placa,marca,modelo,sede
131,NKD501,AKT,NKD 125,Aeropuerto Palonegro
132,XTZ502,Yamaha,XTZ 150 Enduro,Aeropuerto Palonegro
133,VST503,Suzuki,V-Strom 650 XT,Cabecera Principal


### Consulta 2: Ingresos por Segmento Comercial de Clientes
**Problema de Negocio:** Evaluar el rendimiento financiero de cada segmento de mercado (Empresas corporativas, Clientes frecuentes con membresía activa, y Clientes ocasionales).
**Técnica SQL:** Clasificación con expresión `CASE WHEN`, `EXISTS`, agregación `SUM(p.valor)` y agrupación `GROUP BY`.


In [1]:
SELECT
  CASE
    WHEN e.id_cliente IS NOT NULL THEN 'EMPRESARIAL'
    WHEN EXISTS (SELECT 1 FROM membresia m
                 WHERE m.id_cliente = r.id_cliente AND m.estado = 'ACTIVA') THEN 'FRECUENTE'
    ELSE 'OCASIONAL'
  END AS tipo_comercial,
  COUNT(DISTINCT p.id_reserva) AS reservas_pagadas,
  SUM(p.valor) AS ingresos_totales
FROM pago p
JOIN reserva r     ON r.id_reserva = p.id_reserva
LEFT JOIN empresa e ON e.id_cliente = r.id_cliente
WHERE p.estado = 'APLICADO'
GROUP BY tipo_comercial
ORDER BY ingresos_totales DESC;


tipo_comercial,reservas_pagadas,ingresos_totales
EMPRESARIAL,8,24312327.0
FRECUENTE,13,14180945.0
OCASIONAL,12,9675840.0


### Consulta 3: Trazabilidad Completa de Cliente (Visión 360°)
**Problema de Negocio:** El área de servicio al cliente requiere auditar el historial integral de la cliente frecuente Laura Gómez (`id_cliente = 200`), mostrando reservas, contratos asociados, estados y montos facturados.
**Técnica SQL:** Encadenamiento de `LEFT JOIN` entre `reserva`, `categoria`, `contrato` y `factura` para preservar reservas incluso sin contrato o sin factura aún.


In [1]:
SELECT
  r.id_reserva, r.codigo_reserva, r.estado AS estado_reserva,
  r.fecha_hora_recogida, r.fecha_hora_devolucion_prevista,
  cat.nombre AS categoria,
  co.numero_contrato, co.estado AS estado_contrato,
  f.numero_factura, (f.subtotal + f.impuesto) AS total_pagado
FROM reserva r
JOIN categoria cat      ON cat.id_categoria = r.id_categoria
LEFT JOIN contrato co    ON co.id_reserva = r.id_reserva
LEFT JOIN factura f      ON f.id_contrato = co.id_contrato
WHERE r.id_cliente = 200
ORDER BY r.fecha_hora_recogida;


id_reserva,codigo_reserva,estado_reserva,fecha_hora_recogida,fecha_hora_devolucion_prevista,categoria,numero_contrato,estado_contrato,numero_factura,total_pagado
500,R-2026-0500,CUMPLIDA,2026-05-10 10:00:00,2026-05-13 10:00:00,Económico Compacto,C-2026-0900,CERRADO,FAC-000950,459637.5
501,R-2026-0501,CUMPLIDA,2026-08-15 09:00:00,2026-08-19 09:00:00,SUV Familiar,C-2026-0901,CERRADO,FAC-000951,967470.0
502,R-2026-0502,CUMPLIDA,2026-10-10 08:30:00,2026-10-13 08:30:00,Económico Compacto,C-2026-0902,CERRADO,FAC-000952,459637.5
528,R-2026-0528,CONFIRMADA,2026-11-08 09:00:00,2026-11-12 18:00:00,Económico Compacto,C-2026-0928,ABIERTO,NaN,NaN


### Consulta 4: Top 5 Vehículos con Mayor Retorno y Kilómetros Generados
**Problema de Negocio:** La gerencia de operaciones requiere conocer qué unidades presentan mayor rotación comercial y cuántos kilómetros efectivos han recorrido en contratos completados.
**Técnica SQL:** Agregación `COUNT`, cálculo aritmético entre inspección de devolución y entrega `(di.km_lectura - ei.km_lectura)` y ordenamiento descendente con `LIMIT 5`.


In [1]:
SELECT
  v.id_vehiculo, v.placa, m.marca, m.nombre AS modelo,
  COUNT(co.id_contrato) AS veces_rentado,
  SUM(di.km_lectura - ei.km_lectura) AS km_totales_generados
FROM vehiculo v
JOIN modelo m ON m.id_modelo = v.id_modelo
JOIN contrato co ON co.id_vehiculo = v.id_vehiculo
LEFT JOIN inspeccion ei ON ei.id_contrato = co.id_contrato AND ei.tipo = 'ENTREGA'
LEFT JOIN inspeccion di ON di.id_contrato = co.id_contrato AND di.tipo = 'DEVOLUCION'
WHERE co.estado = 'CERRADO'
GROUP BY v.id_vehiculo, v.placa, m.marca, m.nombre
ORDER BY veces_rentado DESC, km_totales_generados DESC
LIMIT 5;


id_vehiculo,placa,marca,modelo,veces_rentado,km_totales_generados
123,HLX303,Toyota,Hilux Doble Cabina 4x4,2,4230
121,HLX301,Toyota,Hilux Doble Cabina 4x4,2,4200
110,DST201,Renault,Duster Zen 4x2,2,1800
100,KWD101,Renault,Kwid Zen,2,930
122,FRN302,Nissan,Frontier Pro4X,1,3350


### Consulta 5: Tasa de Ocupación y Utilización de Flota por Sede
**Problema de Negocio:** Determinar la eficiencia en el uso de la flota física asignada a cada sede del país, calculando el porcentaje de vehículos que han generado contratos cerrados.
**Técnica SQL:** Conteo condicional `COUNT(DISTINCT ...)`, cálculo porcentual redondeado con `ROUND` y ordenamiento de mayor a menor productividad.


In [1]:
SELECT
  s.nombre AS sede,
  COUNT(DISTINCT v.id_vehiculo) AS vehiculos_en_sede,
  COUNT(DISTINCT co.id_vehiculo) AS vehiculos_con_alquiler,
  ROUND(100.0 * COUNT(DISTINCT co.id_vehiculo) / COUNT(DISTINCT v.id_vehiculo), 1) AS pct_utilizacion
FROM sede s
JOIN vehiculo v ON v.id_sede_actual = s.id_sede
LEFT JOIN contrato co ON co.id_vehiculo = v.id_vehiculo AND co.estado = 'CERRADO'
GROUP BY s.id_sede, s.nombre
ORDER BY pct_utilizacion DESC;


sede,vehiculos_en_sede,vehiculos_con_alquiler,pct_utilizacion
Aeropuerto Bonilla Aragón,3,3,100.0
Aeropuerto Rafael Núñez,3,3,100.0
Aeropuerto Palonegro,8,7,87.5
Aeropuerto J.M. Córdova,7,4,57.1
Cabecera Principal,4,2,50.0
Aeropuerto El Dorado T1,10,5,50.0
Bogotá Norte - Usaquén,3,0,0.0
Medellín El Poblado,3,0,0.0
Aeropuerto E. Cortissoz,1,0,0.0


### Consulta 6: Ranking de Clientes en Programa de Fidelización
**Problema de Negocio:** La dirección de marketing necesita identificar los clientes con mayor puntaje acumulado para invitaciones VIP y upgrades de categoría.
**Técnica SQL:** Subconsulta escalar correlacionada con `SUM(mp.puntos)`, función `COALESCE` para manejar clientes sin movimientos, y vinculación con planes de membresía vigentes.


In [1]:
SELECT
  pn.nombres, pn.apellidos,
  COALESCE((SELECT SUM(mp.puntos) FROM movimiento_puntos mp
            WHERE mp.id_cliente = pn.id_cliente), 0) AS saldo_puntos,
  pm.nombre AS plan_actual
FROM persona_natural pn
LEFT JOIN membresia mb       ON mb.id_cliente = pn.id_cliente AND mb.estado = 'ACTIVA'
LEFT JOIN plan_membresia pm  ON pm.id_plan = mb.id_plan
ORDER BY saldo_puntos DESC
LIMIT 5;


nombres,apellidos,saldo_puntos,plan_actual
María,Fernández Soto,6400,PLATINO
Laura,Gómez Prada,4200,ORO
Gabriela,Torres Saldarriaga,3800,PLATINO
Camila,Suárez Duque,3200,ORO
Andrés,Rojas Pinzón,2100,PLATA


### Consulta 7: Matriz de Ingresos por Categoría Vehicular y Ciudad
**Problema de Negocio:** Identificar qué categorías de vehículos producen mayor facturación en cada ciudad, guiando las decisiones de reubicación y compra de flota.
**Técnica SQL:** `JOIN` multidimensional entre facturas, contratos, reservas, categorías y sedes, agrupando por categoría y ciudad.


In [1]:
SELECT
  cat.nombre AS categoria, c.nombre AS ciudad,
  COUNT(f.id_factura) AS contratos_facturados,
  SUM(f.subtotal + f.impuesto) AS ingresos
FROM factura f
JOIN contrato co ON co.id_contrato = f.id_contrato
JOIN reserva r    ON r.id_reserva = co.id_reserva
JOIN categoria cat ON cat.id_categoria = r.id_categoria
JOIN sede s        ON s.id_sede = r.id_sede_recogida
JOIN ciudad c      ON c.id_ciudad = s.id_ciudad
GROUP BY cat.nombre, c.nombre
ORDER BY ingresos DESC;


categoria,ciudad,contratos_facturados,ingresos
Pickup 4x4 Doble Cabina,Bucaramanga,3,9700642.0
Pickup 4x4 Doble Cabina,Bogotá,2,6183954.0
Pickup 4x4 Doble Cabina,Cali,1,4808790.0
SUV Familiar,Bucaramanga,3,3630095.0
Van Pasajeros 9P,Bucaramanga,1,2330496.0
Van Pasajeros 9P,Cartagena,1,2197692.0
Van Pasajeros 9P,Bogotá,1,2023000.0
Van Pasajeros 9P,Medellín,1,1868300.0
SUV Familiar,Medellín,1,1800113.0
SUV Familiar,Cartagena,1,1731450.0


### Consulta 8: Gestión de Flota en Taller y Días Fuera de Servicio
**Problema de Negocio:** Monitorear diariamente los vehículos que se encuentran actualmente inoperativos en taller, conociendo la avería técnica y los días que llevan retenidos.
**Técnica SQL:** Filtrado de mantenimientos activos (`fecha_fin_real IS NULL`) y cálculo dinámico de días transcurridos mediante `DATEDIFF(CURDATE(), mt.fecha_inicio)`.


In [1]:
SELECT
  v.placa, m.marca, m.nombre AS modelo, mt.tipo, mt.descripcion,
  mt.fecha_inicio, DATEDIFF(CURDATE(), mt.fecha_inicio) AS dias_en_taller
FROM mantenimiento mt
JOIN vehiculo v ON v.id_vehiculo = mt.id_vehiculo
JOIN modelo m   ON m.id_modelo = v.id_modelo
WHERE mt.fecha_fin_real IS NULL
ORDER BY dias_en_taller DESC;


placa,marca,modelo,tipo,descripcion,fecha_inicio,dias_en_taller
MCX209,Mazda,CX-5 Touring 2.0,CORRECTIVO,Falla en compresor aire acondicionado y fuga de refrigerante,2026-10-26 08:00:00,15
FRN304,Nissan,Frontier Pro4X,CORRECTIVO,Reparación de turbo y cambio de inyectores diésel,2026-10-31 08:00:00,10
VAN403,Hyundai,H1 Minibus 9 Pasajeros,CORRECTIVO,Reparación de módulo de transmisión y suspensión delantera,2026-11-04 09:00:00,6


### Consulta 9: Análisis de Penalizaciones y Frecuencia de Cobros Extra
**Problema de Negocio:** Auditar el impacto de penalizaciones (daños mecánicos, retrasos, combustible faltante, limpiezas especiales, cancelaciones y no-shows).
**Técnica SQL:** Agrupación por tipo de cargo con métricas combinadas: conteo total, recaudación agregada `SUM` y costo promedio `AVG`.


In [1]:
SELECT
  tipo, COUNT(*) AS cantidad, SUM(valor) AS valor_total, ROUND(AVG(valor), 0) AS valor_promedio
FROM cargo
GROUP BY tipo
ORDER BY valor_total DESC;


tipo,cantidad,valor_total,valor_promedio
DANO,2,610000,305000.0
CANCELACION,2,240000,120000.0
RETORNO_OTRA_SEDE,1,195000,195000.0
LIMPIEZA,1,80000,80000.0
NO_SHOW,1,70000,70000.0
DEVOLUCION_TARDIA,1,65000,65000.0
COMBUSTIBLE,1,65000,65000.0


### Consulta 10: Alerta Preventiva de Vencimiento de Licencias de Conducción
**Problema de Negocio:** Prevención legal y comercial: detectar clientes cuyas licencias expirarán en los próximos 60 días para contactarlos antes de que reserven y evitar rechazos en mostrador.
**Técnica SQL:** Comparación de fechas con `CURDATE()`, cálculo de días restantes con `DATEDIFF` y ordenamiento prioritario por urgencia de vencimiento.


In [1]:
SELECT
  pn.nombres, pn.apellidos, c.correo, l.categoria, l.fecha_vencimiento,
  DATEDIFF(l.fecha_vencimiento, CURDATE()) AS dias_para_vencer
FROM licencia l
JOIN persona_natural pn ON pn.id_cliente = l.id_cliente
JOIN cliente c ON c.id_cliente = pn.id_cliente
WHERE l.fecha_vencimiento <= '2027-01-09' AND l.fecha_vencimiento >= CURDATE()
ORDER BY dias_para_vencer;


nombres,apellidos,correo,categoria,fecha_vencimiento,dias_para_vencer
Laura,Gómez Prada,laura.gomez@corporativo-andino.com,B1,2026-11-28,18
María,Fernández Soto,maria.fernandez@gmail.com,B1,2026-12-15,35
Óscar,Vargas Villamizar,oscar.vargas@agroexport.com,B1,2027-01-02,53
